In [ ]:
# =====================================================================
# STRIDED CONVOLUTIONAL FORWARD-FORWARD GENERATOR (186K params)
# =====================================================================
import os
import math
import numpy as np
import cupy as cp
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader
from tqdm import tqdm
from numba import cuda
from safetensors.torch import save_file, load_file
from torchvision.models import vgg16, VGG16_Weights


# =====================================================================
# STRIDED DECODER KERNEL (supports stride=1 and stride=2)
# =====================================================================
@cuda.jit
def conv_karl_update_kernel(G, z_current, x_true, x_pred, lr, weight_decay, stride, padding):
    z_c_idx, x_c_idx, kh = cuda.grid(3)
    if z_c_idx < G.shape[0] and x_c_idx < G.shape[1] and kh < G.shape[2]:
        for kw in range(G.shape[3]):
            batch_size = z_current.shape[0]
            out_h = x_true.shape[2]
            out_w = x_true.shape[3]
            delta = 0.0

            for b in range(batch_size):
                for oh in range(out_h):
                    for ow in range(out_w):
                        # For transposed conv: oh = zh*stride + kh - padding
                        # So zh = (oh + padding - kh) / stride
                        zh_num = oh + padding - kh
                        zw_num = ow + padding - kw
                        
                        if zh_num % stride == 0 and zw_num % stride == 0:
                            zh = zh_num // stride
                            zw = zw_num // stride
                            if 0 <= zh < z_current.shape[2] and 0 <= zw < z_current.shape[3]:
                                error = x_true[b, x_c_idx, oh, ow] - x_pred[b, x_c_idx, oh, ow]
                                delta += error * z_current[b, z_c_idx, zh, zw]

            current_weight = G[z_c_idx, x_c_idx, kh, kw]
            G[z_c_idx, x_c_idx, kh, kw] = (
                current_weight * (1.0 - weight_decay)
                + (lr / (batch_size * out_h * out_w)) * delta
            )


# =====================================================================
# VGG PERCEPTUAL LOSS (unchanged)
# =====================================================================
class VGGPerceptualLoss(torch.nn.Module):
    def __init__(self, device='cuda'):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        self.feature_extractor = vgg.features[:16].to(device).eval()
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device))

    def normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, x, x_hat):
        phi_x = self.feature_extractor(self.normalize(x))
        phi_x_hat = self.feature_extractor(self.normalize(x_hat))
        return F.mse_loss(phi_x, phi_x_hat)


# =====================================================================
# DLPack BRIDGE (unchanged)
# =====================================================================
def cupy_to_torch(cp_arr):
    return torch.from_dlpack(cp_arr)

def torch_to_cupy(th_tensor):
    return cp.from_dlpack(th_tensor)


# =====================================================================
# STRIDED CONV FF LAYER
# =====================================================================
class CUDAConvFFLayer:
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, lr_rep=0.003, lr_gen=0.01):
        self.in_c = in_channels
        self.out_c = out_channels
        self.ks = kernel_size
        self.stride = stride
        self.lr_rep = lr_rep
        self.lr_gen = lr_gen
        
        # Padding to keep spatial dims clean:
        # stride=1 -> pad=1 (same)
        # stride=2 -> pad=1 (halves spatial size)
        self.padding = 1
        # For decoder stride=2, need output_padding=1 to get exact 2x upsampling with kernel=3
        self.output_padding = 1 if stride == 2 else 0

        # Encoder
        limit_w = np.sqrt(6 / (in_channels * kernel_size * kernel_size + out_channels))
        self.W = cp.random.uniform(-limit_w, limit_w, (out_channels, in_channels, kernel_size, kernel_size), dtype=cp.float32)

        # Decoder (transposed conv)
        self.G = cp.random.uniform(-(limit_w * 2.0), (limit_w * 2.0), (out_channels, in_channels, kernel_size, kernel_size), dtype=cp.float32)

    def forward_encoder(self, x_cupy):
        x_th = cupy_to_torch(x_cupy)
        w_th = cupy_to_torch(self.W)
        with torch.no_grad():
            y_th = F.conv2d(x_th, w_th, stride=self.stride, padding=self.padding)
            y_th = F.leaky_relu(y_th, 0.01)
        return torch_to_cupy(y_th)

    def forward_decoder(self, z_cupy):
        z_th = cupy_to_torch(z_cupy)
        g_th = cupy_to_torch(self.G)
        with torch.no_grad():
            x_pred_th = F.conv_transpose2d(
                z_th, g_th, stride=self.stride, 
                padding=self.padding, output_padding=self.output_padding
            )
            x_pred_th = torch.sigmoid(x_pred_th)
        return torch_to_cupy(x_pred_th)

    def train_encoder(self, x_pos, x_neg, perceptual_fn=None, margin=0.1,
                      w_ff=1.0, w_mse=0.5, w_perc=0.1):
        x_pos_th = cupy_to_torch(x_pos)
        x_neg_th = cupy_to_torch(x_neg)

        W_th = cupy_to_torch(self.W)
        W_th.requires_grad = True
        G_th = cupy_to_torch(self.G).detach()

        # --- POSITIVE ---
        z_pos = F.conv2d(x_pos_th, W_th, stride=self.stride, padding=self.padding)
        z_pos = F.leaky_relu(z_pos, 0.01)
        x_hat_pos = F.conv_transpose2d(
            z_pos, G_th, stride=self.stride, 
            padding=self.padding, output_padding=self.output_padding
        )
        x_hat_pos = torch.sigmoid(x_hat_pos)
        mse_pos = F.mse_loss(x_hat_pos, x_pos_th)

        # --- NEGATIVE ---
        z_neg = F.conv2d(x_neg_th, W_th, stride=self.stride, padding=self.padding)
        z_neg = F.leaky_relu(z_neg, 0.01)
        x_hat_neg = F.conv_transpose2d(
            z_neg, G_th, stride=self.stride,
            padding=self.padding, output_padding=self.output_padding
        )
        x_hat_neg = torch.sigmoid(x_hat_neg)
        mse_neg = F.mse_loss(x_hat_neg, x_neg_th)

        # Goodness = negative MSE
        g_pos = -mse_pos
        g_neg = -mse_neg
        ff_equilibrium = torch.log(1.0 + torch.exp(g_neg - g_pos + margin))

        # Perceptual loss only on RGB (layer 1)
        if perceptual_fn is not None and x_pos_th.shape[1] == 3:
            perc_loss = perceptual_fn(x_pos_th, x_hat_pos)
        else:
            perc_loss = torch.tensor(0.0, device=x_pos_th.device)

        loss = w_ff * ff_equilibrium + w_mse * mse_pos + w_perc * perc_loss
        loss.backward()

        with torch.no_grad():
            grad_cp = torch_to_cupy(W_th.grad)
            self.W -= self.lr_rep * grad_cp

        return (torch_to_cupy(z_pos.detach()), torch_to_cupy(z_neg.detach()),
                torch_to_cupy(ff_equilibrium.detach()), mse_pos.detach(), mse_neg.detach())

    def train_decoder(self, z_current, x_true):
        x_pred = self.forward_decoder(z_current)
        
        # Numba kernel for local decoder update (works for stride=1 and stride=2)
        threads_per_block = (8, 4, 1)
        blocks_x = math.ceil(self.out_c / threads_per_block[0])
        blocks_y = math.ceil(self.in_c / threads_per_block[1])
        blocks_z = math.ceil(self.ks / threads_per_block[2])

        weight_decay = 1e-5
        conv_karl_update_kernel[(blocks_x, blocks_y, blocks_z), threads_per_block](
            self.G, z_current, x_true, x_pred, self.lr_gen, weight_decay,
            self.stride, self.padding
        )

        mse = cp.mean((x_true - x_pred)**2)
        return mse, x_pred


# =====================================================================
# BOTTLENECKED GENERATOR (186K params, 8x spatial compression)
# =====================================================================
class AnimeForwardForwardGenerator:
    def __init__(self):
        # 64x64 -> 32x32 -> 16x16 -> 8x8 bottleneck
        self.enc_layer1 = CUDAConvFFLayer(in_channels=3,  out_channels=32,  stride=2)
        self.enc_layer2 = CUDAConvFFLayer(in_channels=32, out_channels=64,  stride=2)
        self.enc_layer3 = CUDAConvFFLayer(in_channels=64, out_channels=128, stride=2)
        
        # Running stats for latent z3 (fitted during training, used for generation)
        self.z3_mean = cp.zeros((128, 8, 8), dtype=cp.float32)
        self.z3_var  = cp.ones((128, 8, 8), dtype=cp.float32)
        self.z3_count = 0

    def update_z3_stats(self, z3_batch):
        """Welford's online algorithm for latent statistics"""
        batch_mean = cp.mean(z3_batch, axis=0)
        batch_var = cp.var(z3_batch, axis=0)
        n = z3_batch.shape[0]
        
        if self.z3_count == 0:
            self.z3_mean = batch_mean
            self.z3_var = batch_var
            self.z3_count = n
        else:
            delta = batch_mean - self.z3_mean
            total = self.z3_count + n
            self.z3_mean = (self.z3_mean * self.z3_count + batch_mean * n) / total
            # Combine variances
            m_a = self.z3_var * self.z3_count
            m_b = batch_var * n
            self.z3_var = (m_a + m_b + (delta**2) * self.z3_count * n / total) / total
            self.z3_count = total

    def train_step(self, x_pos, x_neg, perceptual_fn=None):
        z1_pos, z1_neg, ff1, mp1, mn1 = self.enc_layer1.train_encoder(x_pos, x_neg, perceptual_fn=perceptual_fn)
        z2_pos, z2_neg, ff2, mp2, mn2 = self.enc_layer2.train_encoder(z1_pos, z1_neg)
        z3_pos, z3_neg, ff3, mp3, mn3 = self.enc_layer3.train_encoder(z2_pos, z2_neg)

        # Update running stats on REAL latent codes
        self.update_z3_stats(z3_pos)

        # Decoder pass
        mse3, z2_pred = self.enc_layer3.train_decoder(z_current=z3_pos, x_true=z2_pos)
        mse2, z1_pred = self.enc_layer2.train_decoder(z_current=z2_pos, x_true=z1_pos)
        mse1, x_pred  = self.enc_layer1.train_decoder(z_current=z1_pos, x_true=x_pos)

        total_mse = mse1 + mse2 + mse3
        total_ff = ff1 + ff2 + ff3
        return x_pred, total_mse, total_ff, (mp1+mp2+mp3), (mn1+mn2+mn3)

    def save_safetensors(self, filepath):
        tensors = {
            "layer1.W": torch.from_numpy(self.enc_layer1.W.get()),
            "layer1.G": torch.from_numpy(self.enc_layer1.G.get()),
            "layer2.W": torch.from_numpy(self.enc_layer2.W.get()),
            "layer2.G": torch.from_numpy(self.enc_layer2.G.get()),
            "layer3.W": torch.from_numpy(self.enc_layer3.W.get()),
            "layer3.G": torch.from_numpy(self.enc_layer3.G.get()),
            "z3.mean":  torch.from_numpy(self.z3_mean.get()),
            "z3.std":   torch.from_numpy(cp.sqrt(self.z3_var).get()),
        }
        save_file(tensors, filepath)
        print(f"\nModel saved to {filepath}")

    def load_safetensors(self, filepath):
        tensors = load_file(filepath)
        self.enc_layer1.W = cp.array(tensors["layer1.W"].numpy())
        self.enc_layer1.G = cp.array(tensors["layer1.G"].numpy())
        self.enc_layer2.W = cp.array(tensors["layer2.W"].numpy())
        self.enc_layer2.G = cp.array(tensors["layer2.G"].numpy())
        self.enc_layer3.W = cp.array(tensors["layer3.W"].numpy())
        self.enc_layer3.G = cp.array(tensors["layer3.G"].numpy())
        self.z3_mean = cp.array(tensors["z3.mean"].numpy())
        self.z3_var = cp.array(tensors["z3.std"].numpy()) ** 2
        print(f"Loaded {filepath}. Ready to generate.")


# =====================================================================
# NEGATIVE DATA GENERATOR (unchanged)
# =====================================================================
def make_negative(x, noise_scale=1.0, patch_size=2, replace_ratio=0.5):
    B, C, H, W = x.shape
    x_neg = np.copy(x)
    for b in range(B):
        for c in range(C):
            patches = []
            for i in range(0, H, patch_size):
                for j in range(0, W, patch_size):
                    patches.append(x_neg[b, c, i:i+patch_size, j:j+patch_size].copy())
            np.random.shuffle(patches)
            idx = 0
            for i in range(0, H, patch_size):
                for j in range(0, W, patch_size):
                    if np.random.rand() < replace_ratio:
                        static = np.random.normal(loc=0.5, scale=noise_scale, size=(patch_size, patch_size))
                        x_neg[b, c, i:i+patch_size, j:j+patch_size] = static
                    else:
                        x_neg[b, c, i:i+patch_size, j:j+patch_size] = patches[idx]
                    idx += 1
    return np.clip(x_neg, 0.0, 1.0)


# =====================================================================
# GENERATION SCRIPT (no dataset, no pixel optimization)
# =====================================================================
def generate(model_path, out_path="generated.png", n_samples=16, device='cuda'):
    model = AnimeForwardForwardGenerator()
    model.load_safetensors(model_path)
    
    # Sample from learned latent distribution
    z3_mean = cupy_to_torch(model.z3_mean)
    z3_std  = cupy_to_torch(cp.sqrt(model.z3_var))
    
    with torch.no_grad():
        # Sample z3 ~ N(mean, std)
        z3 = torch.randn(n_samples, 128, 8, 8, device=device) * z3_std + z3_mean
        
        # Cascade decode (single forward pass, no loops, no gradients)
        z2 = F.conv_transpose2d(z3, cupy_to_torch(model.enc_layer3.G).to(device), 
                                stride=2, padding=1, output_padding=1)
        z2 = torch.sigmoid(z2)
        
        z1 = F.conv_transpose2d(z2, cupy_to_torch(model.enc_layer2.G).to(device),
                                stride=2, padding=1, output_padding=1)
        z1 = torch.sigmoid(z1)
        
        x = F.conv_transpose2d(z1, cupy_to_torch(model.enc_layer1.G).to(device),
                               stride=2, padding=1, output_padding=1)
        x = torch.sigmoid(x)
        
        vutils.save_image(x, out_path, nrow=4, normalize=False)
        print(f"Generated {n_samples} images to {out_path}")


# =====================================================================
# TRAINING SCRIPT (drop-in replacement)
# =====================================================================
if __name__ == "__main__":
    import glob, random
    from PIL import Image
    from torch.utils.data import Dataset

    class MultiSourceFlatImageFolder(Dataset):
        def __init__(self, sources, transform=None):
            self.transform = transform
            self.image_paths = []
            for source_dict in sources:
                root = source_dict['path']
                limit = source_dict.get('limit', None)
                paths = (
                    glob.glob(os.path.join(root, '**', '*.jpg'), recursive=True) + 
                    glob.glob(os.path.join(root, '**', '*.png'), recursive=True) + 
                    glob.glob(os.path.join(root, '**', '*.jpeg'), recursive=True)
                )
                if limit and len(paths) > limit:
                    random.shuffle(paths)
                    paths = paths[:limit]
                self.image_paths.extend(paths)
            random.shuffle(self.image_paths)

        def __len__(self):
            return len(self.image_paths)

        def __getitem__(self, idx):
            try:
                image = Image.open(self.image_paths[idx]).convert('RGB')
            except Exception:
                image = Image.new('RGB', (64, 64))
            if self.transform:
                image = self.transform(image)
            return image, 0

    ganyu_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    dataset_sources = [
        {'path': '/kaggle/input/datasets/andy8744/ganyu-genshin-impact-anime-faces-gan-training/ganyu'},
        {'path': '/kaggle/input/datasets/splcher/animefacedataset/images', 'limit': 3000}
    ]
    
    train_dataset = MultiSourceFlatImageFolder(sources=dataset_sources, transform=ganyu_transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    perceptual_fn = VGGPerceptualLoss(device='cuda' if torch.cuda.is_available() else 'cpu')
    model = AnimeForwardForwardGenerator()
    
    model_path = "anime_rgl_ae_bottleneck.safetensors"
    if os.path.exists(model_path):
        model.load_safetensors(model_path)

    print(f"Training on {len(train_dataset)} images | 186K params | 8x bottleneck")
    EPOCHS = 1
    
    for epoch in range(EPOCHS):
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch_idx, (data, _) in enumerate(train_bar):
            x_pos = cp.array(data.numpy())
            x_neg = cp.array(make_negative(x_pos.get()))
            
            x_pred, gen_loss, ff_eq, mse_pos, mse_neg = model.train_step(
                x_pos, x_neg, perceptual_fn=perceptual_fn
            )
            train_bar.set_postfix({
                'Gen Loss': f'{gen_loss.get():.4f}',
                'FF Eq': f'{ff_eq.get():.4f}',
                'mse_pos': f'{mse_pos.item():.4f}',
                'mse_neg': f'{mse_neg.item():.4f}'
            })
            
        # Reconstruction check
        orig = torch.from_numpy(x_pos[:16].get())
        recon = torch.from_numpy(x_pred[:16].get())
        vutils.save_image(torch.cat([orig, recon], dim=0), 
                         f'recon_epoch_{epoch+1}.png', nrow=16)
        
        # ACTUAL GENERATION (sample from learned latent prior)
        generate(model_path, out_path=f'generated_epoch_{epoch+1}.png', n_samples=16)
        
    model.save_safetensors(model_path)
